In [ ]:
# ---- 1. CONFIG (paste the values printed by setup_server.sh) ----
TUNNEL_URL  = ""    # chisel server + Caddy on the VPS
LLM_URL     = ""  # public URL: Caddy sends it to the hub (not to a tunnel port any more)
TUN_USER    = "nb"
TUN_PASS    = "CHANGE_ME"       # must match /etc/chisel/secrets.env on the VPS
LLAMA_KEY   = "sk-CHANGE_ME"             # only used between the hub and llama-server; any secret string

# ---- NEW in v35: this notebook's identity on the hub (llm1khan). Ports are picked automatically. ----
SERVER_NAME = "sr1"     # UNIQUE per notebook (sr1, sr2, ...). Clients call it as "<SERVER_NAME>/<MODEL_NAME>"
MODEL_NAME  = "model1"  # short model name clients use; "" = use the GGUF file name
HUB_KEY     = "CHANGE_ME-hub-admin"   # must match HUB_ADMIN_KEY of the hub on the VPS

LLAMA_VARIANT = "auto"  # mainline backend only: "auto" | "cuda13-portable" | "cuda12-portable" | "cpu" ...
GPU_LAYERS  = "auto"    # "auto" = --fit on, exactly like Unsloth Studio (recommended) | or a fixed number
PARALLEL    = 4         # Unsloth Studio default
SPEC_DEFAULT = True     # --spec-default (n-gram speculation, no draft model needed) - what Unsloth Studio uses
LOCAL_PORT  = 8090      # first local port to try; moves to the next free one automatically (8080 is taken by marimo)
HF_TOKEN    = ""        # leave empty for public repos

# ---- 1b. WHAT TO RUN: just change these lines ----
HF_MODEL = ""   # Hugging Face repo (GATED: accept the terms on the HF page + set HF_TOKEN above)
HF_FILE  = ""   # exact .gguf name; or leave "" and set QUANT below
QUANT    = "Q4_K_M"      # used only when HF_FILE is empty (picks the matching file)

ENGINE    = "auto"       # auto = prism fork if the name has "bonsai", else mainline | or force: "mainline" "prism" "git" "url"
MTP       = False        # auto = on when "mtp" is in the model name | True | False
MTP_DRAFT = 2            # draft tokens (dense models 2, MoE 3)
# ---- speculative decoding (pick ONE draft source; the cell below combines it with ngram-mod) ----
DRAFT_FILE = ""  # DFlash/DFlash2 draft GGUF in the same HF repo; "" = off
DRAFT_N    = 7           # DFlash draft tokens: 7 for code/math, ~3 for open-ended text (15 is usually slower)
NGRAM_MOD  = True        # add ngram-mod on top (copies text already seen: tool output, files). Auto-dropped if the build rejects it
SPLIT_MODE = "layer"     # multi-GPU: "layer" (safe) | "tensor" (experimental, needs KV f16, disables --fit)
KV        = "f16"        # "f16" fastest | "q8_0" half the KV memory (for long context)
CTX       = 32768
EXTRA_ARGS = []          # any extra llama-server flags, e.g. ["--temp", "0.7"]

# only for ENGINE = "git" (compile any fork/branch, e.g. Mira) or "url" (any prebuilt .tar.gz)
GIT_URL, GIT_BRANCH, TARBALL_URL = "", "", ""

# previous setup: HF_MODEL="ggml-org/Qwen3.6-27B-MTP-GGUF"; HF_FILE=""; QUANT="Q4_K_M"; MTP="auto"; DRAFT_FILE=""
# ---- copy-paste examples (replace the lines above) ----
# your old model, no MTP:   HF_MODEL="SC117/Qwen3.8-27B-Uncensored-FIT-GGUF"; HF_FILE="Qwen3.8-27B-Uncensored-FIT-13.5G-IQ4_XS.gguf"
# Bonsai 27B (prism fork):  HF_MODEL="prism-ml/Ternary-Bonsai-2-27B-gguf";    HF_FILE="Ternary-Bonsai-2-27B-PQ2_0.gguf"
# Bonsai 9B + MTP:          HF_MODEL="prism-ml/Ternary-Bonsai-2-9B-gguf";     HF_FILE="Ternary-Bonsai-2-9B-PQ2_0-MTP-Q8_0.gguf"   (repo name unverified)
# Mira / other fork:        ENGINE="git"; GIT_URL="https://github.com/OWNER/llama.cpp"; GIT_BRANCH="BRANCH"; HF_MODEL="OWNER/MODEL-GGUF"


In [ ]:
# ---- 2. GPU info (no installs needed: llama.cpp comes as a prebuilt bundle) ----
import subprocess
print(subprocess.run("nvidia-smi", shell=True, text=True, capture_output=True).stdout[:1200]
      or "nvidia-smi not found -> no NVIDIA GPU visible (CPU only)")


In [ ]:
# ---- 3. llama.cpp for the chosen PROFILE: mainline (Unsloth) / prism (Bonsai) / git build / url ----
def _install_llama():
    import glob, os, re, shutil, subprocess, tarfile, time, urllib.request, platform

    name = (HF_MODEL + HF_FILE).lower()
    eng = ENGINE if ENGINE != "auto" else ("prism" if "bonsai" in name else "mainline")
    use_mtp = ("mtp" in name) if MTP == "auto" else bool(MTP)
    cfg = dict(backend=eng, repo=HF_MODEL, file=HF_FILE, quant=QUANT, ctx=CTX, kv=KV, np=PARALLEL,
               spec="mtp" if use_mtp else "none", spec_n=MTP_DRAFT, extra=list(EXTRA_ARGS),
               git_url=GIT_URL, git_branch=GIT_BRANCH, url=TARBALL_URL)
    be = cfg["backend"]
    print(f"model: {HF_MODEL} | engine: {be} | MTP: {cfg['spec']} (n={cfg['spec_n']}) | kv: {cfg['kv']} | ctx: {cfg['ctx']}")

    W = "/tmp/llm"; os.makedirs(W, exist_ok=True)
    arch = {"x86_64": "x64", "amd64": "x64", "aarch64": "arm64", "arm64": "arm64"}[platform.machine()]

    def out(cmd):
        r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
        return (r.stdout or "") + (r.stderr or "")

    def parse_driver_cuda(text):
        m = re.search(r"(?:UMD|CUDA) Version:\s*(\d+)\.(\d+)", text)
        return (int(m.group(1)), int(m.group(2))) if m else None

    smi = out("nvidia-smi") if shutil.which("nvidia-smi") else ""
    drv = parse_driver_cuda(smi)
    caps = [c.strip() for c in out("nvidia-smi --query-gpu=compute_cap --format=csv,noheader").splitlines()
            if re.fullmatch(r"\d+\.\d+", c.strip())]
    print(f"driver CUDA: {drv} | compute capability: {caps or 'n/a'} | arch: {arch}")

    def latest_tag(repo_url, fallback):
        for _ in range(3):
            try:
                return urllib.request.urlopen(f"{repo_url}/releases/latest", timeout=20).geturl().rstrip("/").split("/")[-1]
            except Exception as exc:
                err = exc
        print(f"could not resolve latest tag ({err}); using {fallback}")
        return fallback

    def find_server(dst):
        hits = glob.glob(f"{dst}/**/llama-server", recursive=True)
        return hits[0] if hits else None

    def smoke(server):
        os.chmod(server, 0o755)
        d = os.path.dirname(server)
        env = dict(os.environ, LD_LIBRARY_PATH=f"{d}:{os.environ.get('LD_LIBRARY_PATH','')}")
        r = subprocess.run([server, "--list-devices"], text=True, capture_output=True, env=env)
        text = r.stdout + r.stderr
        sees = bool(re.search(r"CUDA\d+:|found \d+ CUDA devices", text))
        return r.returncode == 0 and (sees or not smi), text

    def fetch_bundle(url, dst):
        if find_server(dst): return None
        pkg = f"{W}/{os.path.basename(url)}"
        if not os.path.exists(pkg):
            print("downloading", url)
            r = subprocess.run(f"curl -fSL --retry 3 --retry-all-errors -o {pkg} {url}", shell=True, text=True, capture_output=True)
            if r.returncode: return "download failed: " + r.stderr.strip()[-150:]
        shutil.rmtree(dst, ignore_errors=True); os.makedirs(dst)
        with tarfile.open(pkg) as t: t.extractall(dst)
        os.remove(pkg)
        return None if find_server(dst) else "no llama-server inside the archive"

    # ---------- build candidate list per backend ----------
    cands = []   # (label, url, dst)
    if be == "mainline":
        REPO = "https://github.com/unslothai/llama.cpp"
        if smi and drv:
            variants = (["cuda13-portable", "cuda13-newer", "cuda13-older"] if drv[0] >= 13 else []) \
                       + ["cuda12-portable", "cuda12-newer", "cuda12-older", "cuda12-legacy"]
            if LLAMA_VARIANT != "auto": variants = [LLAMA_VARIANT]
        else:
            variants = [LLAMA_VARIANT if LLAMA_VARIANT != "auto" else "cpu"]
        if arch == "arm64":
            variants = [v for v in variants if v in ("cpu", "vulkan") or "cuda13" in v] or ["cpu"]
        tag = os.environ.get("LLAMA_TAG") or latest_tag(REPO, "b11160-mix-a6922cc")
        print("release:", tag)
        cands = [(v, f"{REPO}/releases/download/{tag}/app-{tag}-linux-{arch}-{v}.tar.gz", f"{W}/bundle-{v}") for v in variants]
    elif be == "prism":
        if not (smi and drv): raise RuntimeError("prism backend needs an NVIDIA GPU here (or use backend='git')")
        REPO = "https://github.com/PrismML-Eng/llama.cpp"
        tag = os.environ.get("PRISM_TAG") or latest_tag(REPO, "prism-b10777-e3de715")
        print("release:", tag)
        for ver in ("13.3", "13.1", "12.8", "12.4"):          # newest build the driver can run first
            if tuple(int(x) for x in ver.split(".")) <= drv:
                cands.append((f"cuda{ver}", f"{REPO}/releases/download/{tag}/llama-{tag}-bin-linux-cuda-{ver}-x64.tar.gz",
                              f"{W}/prism-{tag}-cuda{ver}"))
    elif be == "url":
        u = cfg["url"]; cands = [("url", u, f"{W}/url-{abs(hash(u)) % 10**8}")]
    elif be == "git":
        url, br = cfg["git_url"], cfg.get("git_branch", "")
        if "OWNER" in url: raise RuntimeError("set GIT_URL (and GIT_BRANCH) in the config cell")
        slug = re.sub(r"[^A-Za-z0-9]+", "-", url.split("://")[-1] + "-" + br).strip("-")
        src = f"{W}/src-{slug}"; server = f"{src}/build/bin/llama-server"
        if not os.path.exists(server):
            nvcc = shutil.which("nvcc") or ("/usr/local/cuda/bin/nvcc" if os.path.exists("/usr/local/cuda/bin/nvcc") else "")
            if not nvcc or not shutil.which("cmake"):
                raise RuntimeError("building needs nvcc + cmake, not found here. Use a prebuilt backend (mainline/prism/url) instead.")
            shutil.rmtree(src, ignore_errors=True)
            subprocess.run(f"git clone --depth 1 {'-b ' + br if br else ''} {url} {src}", shell=True, check=True)
            flags = f"-DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_COMPILER={nvcc} -DLLAMA_CURL=OFF"
            if caps: flags += " -DCMAKE_CUDA_ARCHITECTURES=" + ";".join(sorted({c.replace('.', '') for c in caps}))
            log = f"{W}/build.log"
            print("building (10-25 min the first time)...", flags)
            with open(log, "w") as lf:
                r = subprocess.run(f"cmake -B {src}/build -S {src} {flags}", shell=True, stdout=lf, stderr=subprocess.STDOUT)
                if r.returncode: print(open(log).read()[-1500:]); raise RuntimeError("cmake configure failed")
                p = subprocess.Popen(f"cmake --build {src}/build --target llama-server -j {os.cpu_count()}", shell=True,
                                     stdout=lf, stderr=subprocess.STDOUT)
                while p.poll() is None:
                    time.sleep(60); print("...", open(log).read().strip().splitlines()[-1][:120])
            if p.returncode: print(open(log).read()[-1500:]); raise RuntimeError("build failed")
        ok, text = smoke(server)
        print(text.strip()[-300:])
        if not ok: raise RuntimeError("built llama-server does not see the GPU")
        return server, os.path.dirname(server), cfg
    else:
        raise RuntimeError(f"unknown backend {be!r}")

    tried = []
    for label, url, dst in cands:
        err = fetch_bundle(url, dst)
        if err: tried.append((label, err)); continue
        server = find_server(dst)
        ok, text = smoke(server)
        print(f"[{label}] sees GPU/ok={ok}\n" + text.strip()[-400:])
        if ok: return server, os.path.dirname(server), cfg
        tried.append((label, "no CUDA device: " + text.strip()[-200:]))
    raise RuntimeError("no llama.cpp bundle worked:\n" + "\n".join(f"  {v}: {why}" for v, why in tried)
                     + "\nForce LLAMA_VARIANT (mainline) or paste this output to me.")

LLAMA, LLAMA_LIBS, CFG = _install_llama()
print("llama-server:", LLAMA)


In [ ]:
# ---- 3b. download the model for the PROFILE (resumable; picks the GGUF by quant if no file is named) ----
def _get_model(want=None):
    import json, os, re, subprocess, time, urllib.parse, urllib.request

    repo, want_file, quant = CFG["repo"], (want or CFG.get("file", "")), CFG.get("quant", "")
    W = "/tmp/llm/models"; os.makedirs(W, exist_ok=True)
    hdr = {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}
    req = urllib.request.Request(f"https://huggingface.co/api/models/{repo}?blobs=true", headers=hdr)
    info = json.load(urllib.request.urlopen(req, timeout=60))
    sizes = {s["rfilename"]: (s.get("size") or 0) for s in info.get("siblings", [])}
    ggufs = sorted(n for n in sizes if n.endswith(".gguf") and "mmproj" not in n.lower() and "dflash" not in n.lower())

    if want_file:
        if want_file not in sizes: raise RuntimeError(f"{want_file} not in {repo}. GGUFs there:\n  " + "\n  ".join(ggufs))
        names = [want_file]
    else:
        hits = [n for n in ggufs if quant.lower() in n.lower()]
        if not hits:
            raise RuntimeError(f"no GGUF matching quant {quant!r} in {repo}. Available:\n  " + "\n  ".join(
                f"{n}  ({sizes[n]/1e9:.1f} GB)" for n in ggufs))
        first = hits[0]
        m = re.search(r"-(\d+)-of-(\d+)\.gguf$", first)
        names = sorted(n for n in hits if n.startswith(first[:m.start()])) if m else [first]
        print("picked:", names[0], "(+ shards)" if len(names) > 1 else "")

    total = sum(sizes[n] for n in names)
    smi = subprocess.run("nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits", shell=True,
                         text=True, capture_output=True).stdout.split()
    vram = sum(int(x) for x in smi if x.isdigit()) / 1024
    print(f"model {total/1e9:.1f} GB | total VRAM {vram:.1f} GB")
    if vram and total / 1e9 > vram * 0.9:
        print("WARNING: model barely fits (or does not fit) in VRAM -> pick a smaller quant or expect slow CPU offload")

    for n in names:
        dest = f"{W}/{os.path.basename(n)}"
        if os.path.exists(dest) and os.path.getsize(dest) == sizes[n]:
            continue
        url = f"https://huggingface.co/{repo}/resolve/main/{urllib.parse.quote(n)}"
        cmd = ["curl", "-fL", "-sS", "--retry", "5", "--retry-all-errors", "-C", "-", "-o", dest + ".part", url]
        if HF_TOKEN: cmd[1:1] = ["-H", f"Authorization: Bearer {HF_TOKEN}"]
        p = subprocess.Popen(cmd)
        while p.poll() is None:
            time.sleep(15)
            have = os.path.getsize(dest + ".part") if os.path.exists(dest + ".part") else 0
            print(f"...{os.path.basename(n)}: {have/1e9:.1f}/{sizes[n]/1e9:.1f} GB")
        if p.returncode: raise RuntimeError(f"download of {n} failed (curl exit {p.returncode}); re-run to resume")
        os.rename(dest + ".part", dest)
    return f"{W}/{os.path.basename(names[0])}"

MODEL_PATH = _get_model()
print("model file:", MODEL_PATH)
DRAFT_PATH = _get_model(DRAFT_FILE) if DRAFT_FILE else None
print("draft file:", DRAFT_PATH)


In [ ]:
# ---- 3c. pick ports: a free one on this machine, and a free tunnel port on the VPS (the hub hands it out) ----
def _pick_ports():
    import json, os, socket, urllib.error, urllib.request

    def free_local(start):
        for p in range(start, start + 300):
            with socket.socket() as s:
                try:
                    s.bind(("127.0.0.1", p))
                    return p
                except OSError:
                    continue
        raise RuntimeError(f"no free local port in {start}-{start + 299}")

    lport = free_local(LOCAL_PORT)
    if lport != LOCAL_PORT:
        print(f"local port {LOCAL_PORT} is busy -> using {lport}")

    req = urllib.request.Request(LLM_URL + "/_hub/allocate", json.dumps({"name": SERVER_NAME}).encode(),
                                 {"Authorization": f"Bearer {HUB_KEY}", "Content-Type": "application/json",
                                  "User-Agent": "llama-notebook/35"})
    try:
        with urllib.request.urlopen(req, timeout=20) as resp:
            rport = json.load(resp)["port"]
    except urllib.error.HTTPError as exc:
        raise RuntimeError(f"hub said {exc.code}: {exc.read()[:300].decode('utf-8', 'replace')}\n"
                           f"check HUB_KEY (= HUB_ADMIN_KEY on the VPS) and that SERVER_NAME only uses letters, digits, - and _")
    except Exception as exc:
        raise RuntimeError(f"cannot reach the hub at {LLM_URL}: {exc}\n"
                           f"is llmhub running on the VPS, and does Caddy send this domain to it? (see README.md)")

    model = MODEL_NAME or os.path.basename(MODEL_PATH).rsplit(".gguf", 1)[0]
    print(f"local port {lport} | tunnel port on the VPS {rport} | clients call: {SERVER_NAME}/{model}")
    return {"local": lport, "remote": rport, "model": model}


PORTS = _pick_ports()


In [ ]:
# ---- 4. serve the model: llama-server on 127.0.0.1:lport (MTP / KV options come from the PROFILE) ----
def _run_llama_server():
    import json, os, signal, socket, subprocess, time, urllib.request

    lport = PORTS["local"]
    SERVER_LOG = "/tmp/llm/llama-server.log"
    SERVER_PID = "/tmp/llm/llama-server.pid"

    def tail(path, lines=1):
        try:
            with open(path, "rb") as fh:
                return b"\n".join(fh.read().splitlines()[-lines:]).decode("utf-8", "replace").strip()
        except OSError:
            return ""

    def alive(pid):
        try:
            os.kill(pid, 0)
        except OSError:
            return False
        return True

    if "CHANGE_ME" in LLAMA_KEY or "CHANGE_ME" in TUN_PASS:
        print("warning: LLAMA_KEY / TUN_PASS are still the CHANGE_ME placeholders from setup_server.sh")

    # stop a server this cell started before, so config changes take effect
    old = int(open(SERVER_PID).read().strip() or 0) if os.path.exists(SERVER_PID) else 0
    if old and alive(old):
        print(f"stopping llama-server pid {old}")
        os.kill(old, signal.SIGTERM)
        for _ in range(30):
            if not alive(old):
                break
            time.sleep(1)
        else:
            os.kill(old, signal.SIGKILL)

    env = dict(os.environ)
    env["LD_LIBRARY_PATH"] = os.pathsep.join(
        [p for p in (LLAMA_LIBS, os.path.dirname(LLAMA), env.get("LD_LIBRARY_PATH", "")) if p])
    env["HF_TOKEN"] = HF_TOKEN

    alias = PORTS["model"]   # what clients see after "<SERVER_NAME>/"
    base = [LLAMA, "--host", "127.0.0.1", "--port", str(lport), "--api-key", LLAMA_KEY, "--alias", alias,
            "-m", MODEL_PATH, "--ctx-size", str(CFG["ctx"]), "--parallel", str(CFG["np"]),
            "--flash-attn", "on", "--no-context-shift", "--kv-unified", "--jinja", "--metrics"]
    if GPU_LAYERS == "auto":
        base += ["--fit", "on"]            # same as Unsloth Studio: llama.cpp places layers/ctx across the GPUs
    else:
        base += ["--fit", "off", "--n-gpu-layers", str(GPU_LAYERS)]
    fast = []
    if SPLIT_MODE == "tensor":
        base[base.index("--fit"):base.index("--fit") + 2] = ["--fit", "off", "--n-gpu-layers", "99"]
        base += ["--split-mode", "tensor"]
        if CFG["kv"] != "f16": print("tensor split needs KV f16 -> ignoring KV =", CFG["kv"]); CFG["kv"] = "f16"
    if CFG["kv"] != "f16":
        fast += ["--cache-type-k", CFG["kv"], "--cache-type-v", CFG["kv"]]

    def spec_flags(ngram):
        f, types, n = [], [], CFG["spec_n"]
        if DRAFT_PATH:
            f += ["-md", DRAFT_PATH]; types.append("draft-dflash"); n = DRAFT_N
        elif CFG["spec"] == "mtp":
            types.append("draft-mtp")
            if CFG["backend"] == "prism":
                f += ["--spec-draft-n-min", "0", "--spec-draft-p-min", "0"]
        if types:
            if ngram: types.insert(0, "ngram-mod")
            f += ["--spec-type", ",".join(types), "--spec-draft-n-max", str(n)]
        elif SPEC_DEFAULT:
            f += ["--spec-default"]
        return f

    attempts = [("full", base + fast + spec_flags(NGRAM_MOD) + CFG["extra"])]
    if NGRAM_MOD and (DRAFT_PATH or CFG["spec"] == "mtp"):   # build rejects the combo -> try without ngram-mod
        attempts.append(("no ngram-mod", base + fast + spec_flags(False) + CFG["extra"]))
    attempts.append(("plain (NO speculation / KV options)", base + CFG["extra"]))

    for label, cmd in attempts:
        print(f"[{label}] $", " ".join(cmd))
        with open(SERVER_LOG, "w") as log:
            proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True)
        open(SERVER_PID, "w").write(str(proc.pid))
        print(f"pid {proc.pid} | log {SERVER_LOG}")

        ok, deadline = False, time.time() + 1800
        while time.time() < deadline:
            if proc.poll() is not None:
                break
            try:
                with urllib.request.urlopen(f"http://127.0.0.1:{lport}/health", timeout=5) as resp:
                    if json.load(resp).get("status") == "ok":
                        ok = True
                        break
            except Exception:
                pass
            time.sleep(10)
            print("...", tail(SERVER_LOG, 1)[:160])
        if ok:
            break
        print(tail(SERVER_LOG, 30))
        if proc.poll() is None:
            os.kill(proc.pid, signal.SIGKILL)
        if label != attempts[-1][0]:
            print(f"-> [{label}] failed, trying the next fallback")
    else:
        raise RuntimeError(f"llama-server did not start (log: {SERVER_LOG})")

    req = urllib.request.Request(f"http://127.0.0.1:{lport}/v1/models",
                                 headers={"Authorization": f"Bearer {LLAMA_KEY}"})
    with urllib.request.urlopen(req, timeout=15) as resp:
        models = [m["id"] for m in json.load(resp).get("data", [])]
    print("ready:", f"http://127.0.0.1:{lport}/v1", "| models:", models, "| mode:", label)
    import re
    keys = re.compile(r"offloaded|fit|CUDA\d|found \d+ CUDA|split|flash|spec|draft|nextn|n_ctx|KV buffer", re.I)
    print("---- what the server actually did (check: offloaded N/N layers, both CUDA0 and CUDA1) ----")
    print("\n".join([l for l in open(SERVER_LOG, errors="replace").read().splitlines() if keys.search(l)][:40]))


_run_llama_server()


In [ ]:
import subprocess
print(subprocess.run("du -sh ~/.cache/llama.cpp /root/.cache/llama.cpp 2>/dev/null; ls -la ~/.cache/llama.cpp 2>/dev/null | tail -5; ps -p $(cat /tmp/llm/llama-server.pid) -o pid,etime,stat,cmd | cut -c1-100", shell=True, text=True, capture_output=True).stdout)

In [ ]:
# ---- 5. publish: chisel tunnel on the hub-allocated port, register with the hub, keep re-registering ----
def _publish():
    import json, os, platform, shutil, socket, subprocess, time, urllib.error, urllib.parse, urllib.request

    CHISEL      = "/tmp/llm/chisel"
    CHISEL_TAG  = os.environ.get("CHISEL_TAG", "v1.12.0")
    CHISEL_LOG  = "/tmp/llm/chisel.log"
    CHISEL_PID  = "/tmp/llm/chisel.pid"
    HB_PID      = "/tmp/llm/heartbeat.pid"
    HB_CFG      = "/tmp/llm/heartbeat.json"
    HB_PY       = "/tmp/llm/heartbeat.py"
    lport = PORTS["local"]

    def tail(path, lines=1):
        try:
            with open(path, "rb") as fh:
                return b"".join(fh.readlines()[-lines:]).decode("utf-8", "replace").strip()
        except OSError:
            return ""

    def alive(pid):
        try:
            os.kill(pid, 0)
        except OSError:
            return False
        return True

    def stop(pidfile):
        old = int(open(pidfile).read().strip() or 0) if os.path.exists(pidfile) else 0
        if old and alive(old):
            os.kill(old, 15)
            for _ in range(20):
                if not alive(old):
                    break
                time.sleep(1)

    def hub(path, body=None):
        req = urllib.request.Request(LLM_URL + path, json.dumps(body).encode() if body is not None else None,
                                     {"Authorization": f"Bearer {HUB_KEY}", "Content-Type": "application/json",
                                      "User-Agent": "llama-notebook/35"})
        with urllib.request.urlopen(req, timeout=20) as resp:
            return json.load(resp)

    if not os.path.exists(CHISEL):
        arch = {"x86_64": "amd64", "amd64": "amd64", "aarch64": "arm64", "arm64": "arm64",
                "armv7l": "armv7", "armv6l": "armv6", "i386": "386", "i686": "386"}[platform.machine()]
        asset = f"chisel_{CHISEL_TAG.lstrip('v')}_linux_{arch}.gz"
        url = f"https://github.com/jpillora/chisel/releases/download/{CHISEL_TAG}/{asset}"
        print("downloading", url)
        subprocess.run(f"curl -fSL --retry 3 --retry-all-errors -o /tmp/llm/{asset} {url}", shell=True, check=True)
        subprocess.run(f"gunzip -f /tmp/llm/{asset}", shell=True, check=True)
        shutil.move(f"/tmp/llm/{asset[:-3]}", CHISEL)
        os.chmod(CHISEL, 0o755)
    print("chisel", subprocess.run([CHISEL, "--version"], text=True, capture_output=True).stdout.strip())

    host = urllib.parse.urlparse(TUNNEL_URL).hostname
    try:
        socket.getaddrinfo(host, 443)
    except OSError as exc:
        raise RuntimeError(f"DNS: {host} does not resolve ({exc}). Create an A record for it (DNS only, not proxied) "
                           f"and a Caddy block that proxies to the chisel server, then reload Caddy.")

    def start_chisel(rport):
        stop(CHISEL_PID)
        remote = f"R:127.0.0.1:{rport}:127.0.0.1:{lport}"
        env = dict(os.environ, AUTH=f"{TUN_USER}:{TUN_PASS}")      # keeps the password out of `ps`
        cmd = [CHISEL, "client", "--keepalive", "25s", "--max-retry-interval", "30s", TUNNEL_URL, remote]
        print("$", " ".join(cmd))
        with open(CHISEL_LOG, "w") as log:
            proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True)
        open(CHISEL_PID, "w").write(str(proc.pid))
        deadline = time.time() + 60
        while time.time() < deadline:
            if proc.poll() is not None:
                print(tail(CHISEL_LOG, 20))
                raise RuntimeError(f"chisel client exited with {proc.returncode} (log: {CHISEL_LOG})")
            if "Connected" in open(CHISEL_LOG).read():
                return
            time.sleep(2)
        print(tail(CHISEL_LOG, 5))

    def register(rport):
        body = {"name": SERVER_NAME, "port": rport, "key": LLAMA_KEY, "models": [PORTS["model"]]}
        return hub("/_hub/register", body), body

    rport = PORTS["remote"]
    for attempt in range(1, 4):
        start_chisel(rport)
        res, body = register(rport)
        up = bool(res.get("up"))
        for _ in range(0 if up else 12):                            # hub re-probes every 10 s
            time.sleep(5)
            res, _b = register(rport)
            if res.get("up"):
                up = True
                break
        if up:
            break
        print(f"tunnel port {rport} did not come up (stale tunnel or port in use?) -> asking the hub for another one")
        rport = hub("/_hub/allocate", {"name": SERVER_NAME, "fresh": True})["port"]
        PORTS["remote"] = rport
    else:
        raise RuntimeError("could not publish after 3 ports. chisel log:\n" + tail(CHISEL_LOG, 15)
                           + f"\nCheck that /etc/chisel/users.json allows the reverse remotes (regex must cover the hub's port range).")

    # keep registering every 30 s from a detached process, so the hub forgets nothing after its own restart
    json.dump({"url": LLM_URL, "key": HUB_KEY, "body": body}, open(HB_CFG, "w"))
    open(HB_PY, "w").write(
        "import json, time, urllib.request\n"
        "c = json.load(open('/tmp/llm/heartbeat.json'))\n"
        "while True:\n"
        "    try:\n"
        "        r = urllib.request.Request(c['url'] + '/_hub/register', json.dumps(c['body']).encode(),\n"
        "            {'Authorization': 'Bearer ' + c['key'], 'Content-Type': 'application/json', 'User-Agent': 'llama-notebook/35'})\n"
        "        urllib.request.urlopen(r, timeout=20).read()\n"
        "    except Exception:\n"
        "        pass\n"
        "    time.sleep(30)\n")
    stop(HB_PID)
    hb = subprocess.Popen(["python3", HB_PY], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
    open(HB_PID, "w").write(str(hb.pid))

    print(f"LIVE: {LLM_URL}/v1  ->  model id  {SERVER_NAME}/{PORTS['model']}   (tunnel port {rport} -> local {lport})")


_publish()


In [ ]:
# ---- 6. status: local server, tunnel, heartbeat, and what the hub lists ----
def _status():
    import json, os, urllib.request

    def tail(path, lines=5):
        try:
            with open(path, "rb") as fh:
                return b"".join(fh.readlines()[-lines:]).decode("utf-8", "replace").strip()
        except OSError:
            return "(no log yet)"

    def alive(pid):
        try:
            os.kill(pid, 0)
        except OSError:
            return False
        return True

    lport = PORTS["local"]
    for label, pidfile in (("llama-server", "/tmp/llm/llama-server.pid"), ("chisel", "/tmp/llm/chisel.pid"),
                           ("heartbeat", "/tmp/llm/heartbeat.pid")):
        pid = int(open(pidfile).read().strip() or 0) if os.path.exists(pidfile) else 0
        print(f"{label:13} pid {str(pid or '-'):>8}  {'running' if pid and alive(pid) else 'NOT running'}")

    try:
        req = urllib.request.Request(f"http://127.0.0.1:{lport}/v1/models", headers={"Authorization": f"Bearer {LLAMA_KEY}"})
        with urllib.request.urlopen(req, timeout=10) as resp:
            print("local  ->", [m["id"] for m in json.load(resp).get("data", [])])
    except Exception as exc:
        print("local  ->", type(exc).__name__, exc)
        print(tail("/tmp/llm/llama-server.log", 8))

    try:
        req = urllib.request.Request(LLM_URL + "/_hub/servers", headers={"Authorization": f"Bearer {HUB_KEY}", "User-Agent": "llama-notebook/35"})
        with urllib.request.urlopen(req, timeout=20) as resp:
            servers = json.load(resp)["servers"]
        print("hub    -> servers on the hub:")
        for s in servers:
            me = "  <- this notebook" if s["name"] == SERVER_NAME else ""
            print(f"          {s['name']:10} port {s['port']}  {'UP  ' if s['up'] else 'DOWN'} models {s['models']} active {s['active']}{me}")
        if not any(s["name"] == SERVER_NAME and s["up"] for s in servers):
            print("this notebook is not UP on the hub. check:")
            print("  - the publish cell above finished (chisel log says Connected)")
            print(f"  - /etc/chisel/users.json allows the reverse remote R:127.0.0.1:{PORTS['remote']}")
            print("chisel log:", tail("/tmp/llm/chisel.log", 5))
        else:
            print("endpoint:", LLM_URL + "/v1", "| model id:", f"{SERVER_NAME}/{PORTS['model']}")
    except Exception as exc:
        print("hub    ->", type(exc).__name__, exc)


_status()


In [ ]:
# ---- 7. speed test: tokens/s, and MTP draft acceptance (run it twice; the first call warms up) ----
def _bench():
    import json, urllib.request
    lport = PORTS["local"]
    prompt = "Write a Python function that reads a CSV file and returns a list of dicts, with type hints, a docstring and two usage examples."
    for i in (1, 2):
        body = {"model": "x", "messages": [{"role": "user", "content": prompt}], "max_tokens": 300,
                "temperature": 0, "chat_template_kwargs": {"enable_thinking": False}}
        req = urllib.request.Request(f"http://127.0.0.1:{lport}/v1/chat/completions", json.dumps(body).encode(),
                                     {"Content-Type": "application/json", "Authorization": f"Bearer {LLAMA_KEY}"})
        with urllib.request.urlopen(req, timeout=300) as resp:
            t = json.load(resp).get("timings", {})
        dn, da = t.get("draft_n"), t.get("draft_n_accepted")
        acc = f" | draft accepted {da}/{dn} = {da/dn:.0%}" if dn else " | no speculative drafting active"
        print(f"run {i}: {t.get('predicted_per_second', 0):.1f} tok/s generation, "
              f"{t.get('prompt_per_second', 0):.0f} tok/s prompt{acc}")

_bench()
